# ResNet50 v0 - inferenza RGB
Valutazione dei checkpoint completi salvati da `v0_ResNet50.ipynb`.
Eseguire le celle in ordine dopo aver scelto `CHECKPOINT_NAME`. Il modello non viene
addestrato e i checkpoint non vengono modificati. PNG e manifest sono separati:
il manifest fornisce le etichette, le immagini sono in `data/cifar-10_resized/test`.

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
# Su Colab: montare Drive prima e impostare PROJECT_ROOT alla cartella del progetto.
MODELS_DIR = PROJECT_ROOT / "output" / "models"
CHECKPOINT_NAME = "v0_20epochs_2026-10-16.pth"  # Modificare con il file da valutare.
IMAGE_ROOT = PROJECT_ROOT / "data" / "cifar-10_resized" / "test"
TEST_MANIFEST = PROJECT_ROOT / "output" / "manifest_test.json"
RESULTS_ROOT = PROJECT_ROOT / "runs" / "inference_results"
BATCH_SIZE = 32
EXPORT_TENSORBOARD = False  # True richiede il pacchetto tensorboard.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Checkpoint v0 disponibili:")
for path in sorted(MODELS_DIR.rglob("v0*.pth")):
    print(path.relative_to(MODELS_DIR), "|", path.stat().st_size, "byte")

## 1. Caricamento del checkpoint
Normalizzazione, classi ed epoca provengono dal checkpoint del training v0.
Non si calcolano statistiche sul test. Il solo `state_dict` del vecchio notebook
non contiene questi metadati e non e' il formato previsto.

In [ ]:
CHECKPOINT_PATH = MODELS_DIR / CHECKPOINT_NAME
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
# Un checkpoint compresso non deve essere valutato come RGB.
if checkpoint.get("version") != "v0" or checkpoint.get("representation") not in {
    "uncompressed-rgb-png-v0", "rgb-png-224"
}:
    raise ValueError("Selezionare un checkpoint RGB v0 completo")
VERSION = "v0"
EXPERIMENT = "v0_rgb"
NUM_CLASSES = checkpoint["num_classes"]
idx_to_class = {int(k): v for k, v in checkpoint["idx_to_class"].items()}
CLASS_NAMES = [idx_to_class[i] for i in range(NUM_CLASSES)]
MEAN = checkpoint["normalization"]["mean"]
STD = checkpoint["normalization"]["std"]
if len(MEAN) != 3 or len(STD) != 3 or not np.isfinite([MEAN, STD]).all() or np.any(np.asarray(STD) <= 0):
    raise ValueError("Servono media e std finite per i tre canali RGB, con std positiva")
EPOCH = checkpoint["completed_epochs"]
history = checkpoint.get("history", [])
print("Checkpoint:", CHECKPOINT_PATH)
print("Epoche:", EPOCH, "| media:", MEAN, "| std:", STD)

## 2. Dataset e modello
Stesso preprocessing del test nel training: `ToTensor()` e normalizzazione salvata,
senza flip, resize o crop. I PNG devono essere RGB 224x224.
I nomi `.network.npz` nel manifest vengono associati ai corrispondenti `.png`.
La ResNet50 RGB standard ha la stessa architettura e gli stessi nomi dei pesi della
classe del training: in inferenza basta ripristinare tutto lo state_dict e usare `eval()`.
Non servono download di pesi ImageNet, optimizer o stati casuali del training.

In [ ]:
test_transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

class TestImages(Dataset):
    def __init__(self, manifest_path, image_root):
        self.manifest_path = Path(manifest_path)
        self.image_root = Path(image_root)
        manifest = json.loads(self.manifest_path.read_text(encoding="utf-8"))
        # I due split riutilizzano i nomi delle immagini: evitare etichette del train.
        if manifest["split"] != "test":
            raise ValueError("Usare un manifest del test set")
        mapping = {int(index): name for name, index in manifest["class_to_idx"].items()}
        if mapping != idx_to_class:
            raise ValueError("Le classi del manifest differiscono dal checkpoint")
        self.samples = [dict(file=row["file"].removesuffix(".network.npz") + ".png",
                             label=row["label"]) for row in manifest["samples"]]
        if not self.samples:
            raise ValueError("Il manifest del test e' vuoto")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        row = self.samples[index]
        with Image.open(self.image_root / row["file"]) as image:
            if image.mode != "RGB" or image.size != (224, 224):
                raise ValueError(f"Attesa immagine RGB 224x224: {row['file']}")
            x = test_transform(image)
        return x, row["label"]

test_data = TestImages(TEST_MANIFEST, IMAGE_ROOT)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
model = models.resnet50(weights=None, num_classes=NUM_CLASSES)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()
print("Campioni test:", len(test_data))

## 3. Evaluate the model

Run a forward pass over the test set. Loss is mean cross-entropy per sample. Predictions and probabilities are retained for later analysis and export. No training is performed.

In [ ]:
loss_sum, count = 0.0, 0
confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
probability_batches, target_batches = [], []
with torch.inference_mode():
    for x, y in tqdm(test_loader, desc="Test"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        if not torch.isfinite(logits).all():
            raise ValueError("Logit non finiti")
        loss_sum += nn.functional.cross_entropy(logits, y, reduction="sum").item()
        count += y.numel()
        pred = logits.argmax(dim=1)
        pairs = (y * NUM_CLASSES + pred).cpu().numpy()
        confusion += np.bincount(pairs, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
        probability_batches.append(logits.softmax(dim=1).cpu().numpy())
        target_batches.append(y.cpu().numpy())
test_loss = loss_sum / count
probabilities = np.concatenate(probability_batches)
targets = np.concatenate(target_batches)
predictions = probabilities.argmax(axis=1)
test_source = {"manifest": str(test_data.manifest_path), "image_root": str(test_data.image_root)}
if not np.isfinite(test_loss):
    raise ValueError("Non-finite test loss")

## 4. Metrics and confusion matrices

Rows represent true classes; columns represent predicted classes. The diagonal of the row-normalized matrix gives recall per class.
Precision, recall, and F1 are set to 0 when their denominators are zero. Rows with no samples are marked N/A in the normalized plot. Macro averages include all 10 classes.

In [ ]:
def metrics_from_confusion(cm):
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)
    tp = np.diag(cm).astype(float)
    def divide(a, b):
        return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)
    precision = divide(tp, predicted)
    recall = divide(tp, support)
    f1 = divide(2 * precision * recall, precision + recall)
    normalized = divide(cm.astype(float), support[:, None])
    per_class = pd.DataFrame({"class": CLASS_NAMES, "precision": precision,
                              "recall": recall, "f1": f1, "support": support})
    summary = {"num_samples": int(cm.sum()), "accuracy_pct": float(100 * tp.sum() / cm.sum()),
               "macro_precision": float(precision.mean()), "macro_recall": float(recall.mean()),
               "macro_f1": float(f1.mean()), "weighted_f1": float(np.dot(f1, support) / cm.sum())}
    return summary, per_class, normalized

summary, per_class, normalized = metrics_from_confusion(confusion)
summary.update({"test_loss": test_loss, "epoch": EPOCH, "version": VERSION, "scale": None})
display(pd.DataFrame([summary]))
display(per_class)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, values, title in zip(axes, [confusion, normalized], ["Conteggi", "Recall per riga"]):
    is_normalized = title != "Conteggi"
    im = ax.imshow(values, cmap="Blues", vmin=0, vmax=1 if is_normalized else None)
    fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES),
           xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
           xlabel="Classe predetta", ylabel="Classe vera", title=title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            label = (f"{values[i,j]:.1%}" if confusion[i].sum() else "N/A") if is_normalized else str(values[i,j])
            ax.text(j, i, label, ha="center", va="center", fontsize=8,
                    color="white" if values[i,j] > (values.max() / 2) else "black")
fig.suptitle(f"{EXPERIMENT} — epoca {EPOCH}")
fig.tight_layout()
plt.show()

## 5. Save results

Each execution creates a separate output folder; no checkpoint is overwritten.
`metrics.json` contains provenance information, results, and confusion matrices. CSV files support comparisons across runs.
Per-sample predictions and probabilities are saved in `predictions.npz`.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_NAME = f"{CHECKPOINT_PATH.stem}_inference_{stamp}"
RUN_DIR = RESULTS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
report = {"schema": "resnet-v0-inference-v1", "checkpoint": str(CHECKPOINT_PATH),
          "mode": "inference", "created_utc": stamp,
          "test_source": test_source, "normalization": checkpoint["normalization"],
          "classes": CLASS_NAMES, "summary": summary, "zero_division": 0,
          "confusion_counts": confusion.tolist(), "confusion_proportions": normalized.tolist(),
          "per_class": per_class.to_dict(orient="records"),
          "device": str(DEVICE), "batch_size": BATCH_SIZE,
          "torch_version": str(torch.__version__)}
(RUN_DIR / "metrics.json").write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
pd.DataFrame([{**summary, "checkpoint": CHECKPOINT_PATH.name, "mode": "inference"}]).to_csv(RUN_DIR / "summary.csv", index=False)
per_class.to_csv(RUN_DIR / "per_class.csv", index=False)
for name, values in [("confusion_counts", confusion), ("confusion_proportions", normalized)]:
    pd.DataFrame(values, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RUN_DIR / f"{name}.csv", index_label="true_class")
pd.DataFrame(history).to_csv(RUN_DIR / "training_history.csv", index=False)
fig.savefig(RUN_DIR / "confusion_matrix.png", dpi=160, bbox_inches="tight")
np.savez_compressed(RUN_DIR / "predictions.npz", targets=targets, predictions=predictions,
                    probabilities=probabilities, files=np.asarray([s["file"] for s in test_data.samples]),
                    class_names=np.asarray(CLASS_NAMES))
print("Risultati salvati in:", RUN_DIR)

## 6. Optional TensorBoard export

Set `EXPORT_TENSORBOARD=True` to export the training history and final test results. If the package is missing, install it in your environment with `%pip install tensorboard`.
The imported history preserves epoch numbers, **not the original event timestamps**. The current export logs training loss, accuracy, and learning rate; it does not export epoch durations.
Figures and scalars use shared tags to compare different checkpoints.

In [ ]:
if EXPORT_TENSORBOARD:
    from torch.utils.tensorboard import SummaryWriter
    LOG_ROOT = RESULTS_ROOT / "tensorboard"
    with SummaryWriter(log_dir=str(LOG_ROOT / RUN_NAME)) as writer:
        for entry in history:
            for key, tag in [("loss", "Loss/train"), ("accuracy", "Accuracy_pct/train"), ("lr", "Learning_rate")]:
                if key in entry:
                    writer.add_scalar(tag, entry[key], entry["epoch"])
        writer.add_scalar("Loss/test", test_loss, EPOCH)
        writer.add_scalar("Accuracy_pct/test", summary["accuracy_pct"], EPOCH)
        writer.add_scalar("F1_macro/test", summary["macro_f1"], EPOCH)
        for row in per_class.to_dict(orient="records"):
            for metric in ["precision", "recall", "f1"]:
                writer.add_scalar(f"Per_class/{row['class']}/{metric}", row[metric], EPOCH)
        writer.add_figure("Confusion_matrix/test", fig, EPOCH, close=False)
        writer.add_text("Configuration", json.dumps({"checkpoint": CHECKPOINT_PATH.name,
                        "version": VERSION, "scale": None, "mode": "inference"}, indent=2))
    print("Log TensorBoard:", LOG_ROOT)
else:
    print("Esportazione TensorBoard disattivata; JSON/CSV e figure sono già salvati.")

To open TensorBoard in the notebook, run the following in a new cell:

```python
%load_ext tensorboard
%tensorboard --logdir "{RESULTS_ROOT / 'tensorboard'}"
```

Alternatively, run `tensorboard --logdir <path/to/inference_results/tensorboard>` in a terminal.
Repeat the notebook with other checkpoints to compare runs.
Compare results obtained on the same test samples. Repeated evaluations of a checkpoint are not independent training runs.

Il campo `scale` nei risultati e' sempre `null`, mantenuto solo per allineare
le colonne esportate con v2/v3. Non esiste una scala di compressione per v0.
`accuracy_pct` e' in percentuale; precision, recall e F1 sono frazioni tra 0 e 1.
I file di `predictions.npz` sono nomi PNG, nello stesso ordine del manifest (`shuffle=False`).
Per confrontare esperimenti usare gli stessi campioni test. L'export TensorBoard
usa gli stessi tag e la stessa cartella di v2/v3; l'avvio dell'interfaccia resta manuale.